In [2]:
%%configure
{
    "defaultLakehouse": {
        "name": "lh_oyster_path"
    }
}

StatementMeta(, a4167e5c-85e8-4c01-a3fa-a0861fb129ea, -1, Finished, Available, Finished, True)

In [12]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F




StatementMeta(, 27e99fb3-d386-49c8-94fa-2e5302caffdc, 14, Finished, Available, Finished, False)

In [13]:
# Define the relative path using wildcards to capture all date folders
# This targets all 'Customer' CSV files across any date folder inside 'oyster'
p_root_path = f"Files/landing/oyster/"
p_in_table_name = "Transaction"

StatementMeta(, 27e99fb3-d386-49c8-94fa-2e5302caffdc, 15, Finished, Available, Finished, False)

In [14]:
target_table = f"dbo.ld_oys_{p_in_table_name}".lower()

StatementMeta(, 27e99fb3-d386-49c8-94fa-2e5302caffdc, 16, Finished, Available, Finished, False)

In [15]:

# Build the recursive glob pattern
pattern = f"{p_root_path.rstrip('/')}/**/{p_in_table_name}_*.json"

# Extract only the file paths (keys) to avoid loading file contents into driver memory
source_files = spark.sparkContext.binaryFiles(pattern).keys().collect()

if not source_files:
    raise FileNotFoundError(f"No raw files match {pattern}")

# Sort paths chronologically by the yyyymmddhhmmss folder name and pick the last one
# Note: Extracting the parent folder name (-2) ensures sorting is strictly based on the folder
latest_file_path = sorted(source_files, key=lambda path: path.split("/")[-2])[-1]

print(f"Matched {len(source_files)} file(s).")
print(f"Latest file selected: {latest_file_path}")


StatementMeta(, 27e99fb3-d386-49c8-94fa-2e5302caffdc, 17, Finished, Available, Finished, False)

Matched 1 file(s).
Latest file selected: abfss://5e32bf37-c2be-4676-94c7-97213238ed30@onelake.dfs.fabric.microsoft.com/f27d086e-be7b-4e40-8e62-ea4d210b4538/Files/landing/oyster/20261109142245/Transaction_Ledger_20261109T142245.json


In [16]:
# 1. Read the NDJSON file (omit multiLine="true" because each line is a complete JSON object)
df_raw = spark.read.format("json").load(latest_file_path)

# 2. Flatten the 2D array (array of batches -> single array of transactions) and explode into rows
df_exploded = df_raw.select(
    F.col("SourceSystem"),
    F.col("IngestDate"),
    F.col("Data.AccountID").alias("AccountID"),
    F.explode(F.flatten(F.col("Data.TransactionBatches"))).alias("tx")
)

# 3. Extract the transaction fields into top-level columns and cast types
df_flat = df_exploded.select(
    "SourceSystem",
    "IngestDate",
    "AccountID",
    F.col("tx.TransactionID").alias("TransactionID"),
    F.col("tx.Amount").cast("double").alias("Amount"),
    F.col("tx.Timestamp").cast("timestamp").alias("TransactionTimestamp")
)

# 4. Clean up the file name (removes '?version=...' suffix) and add metadata columns
raw_file_name = F.element_at(F.split(F.input_file_name(), "/"), -1)
clean_file_name = F.element_at(F.split(raw_file_name, "\\?"), 1)

df_with_meta = (
    df_flat.withColumn("meta_load_timestamp_bp", F.from_utc_timestamp(F.current_timestamp(), "Europe/Budapest"))
           .withColumn("meta_load_timestamp_kyiv", F.from_utc_timestamp(F.current_timestamp(), "Europe/Kyiv"))
           .withColumn("meta_load_timestamp_chisinau", F.from_utc_timestamp(F.current_timestamp(), "Europe/Chisinau"))
           .withColumn("meta_source_file", clean_file_name)
)

# 5. Overwrite the managed Delta table
(
    df_with_meta.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Successfully flattened and loaded '{latest_file_path}' into {target_table}")


StatementMeta(, 27e99fb3-d386-49c8-94fa-2e5302caffdc, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8c774db7-eac3-4cae-a6bf-fd019b1cb75d)

SynapseWidget(Synapse.DataFrame, 20f882eb-1de1-48ec-ae97-c55208d8d808)

SynapseWidget(Synapse.DataFrame, 3b977c05-3b45-484f-8183-8e6ebf1aa398)

TrapExitException: exit was called with value '0'.
*** Tips: please refer to https://go.microsoft.com/fwlink/?linkid=2186971 on how to properly stop current session.

In [ ]:
df = spark.sql(f"""SELECT 
    meta_load_timestamp,
    meta_source_file,
    COUNT(*) AS row_count
FROM {target_table}
GROUP BY meta_load_timestamp, meta_source_file
ORDER BY meta_source_file DESC""")

display(df)


StatementMeta(, 27e99fb3-d386-49c8-94fa-2e5302caffdc, -1, Cancelled, , Cancelled, True)